# Example: Stress-Testing the Minimum-Variance Portfolio

In this example, we run systematic "what-if" experiments on the baseline minimum-variance portfolio from the previous example. We'll shift correlations, simulate price drops, increase trading costs, and use Monte Carlo simulation to produce confidence bands around our performance metrics.

> __Learning Objectives:__
>
> * **Correlation Stress Testing**: Implement the crisis-regime blending formula $\tilde{\mathbf{C}} = (1-\alpha)\mathbf{C} + \alpha\,\mathbf{1}\mathbf{1}^{\top}$ and visualize how the eigenstructure collapses
> * **Tail Risk Analysis**: Simulate price drop scenarios and quantify concentration-driven losses via heatmaps
> * **Monte Carlo Confidence Bands**: Generate 1,000 simulated return paths and compute a distribution of portfolio outcomes (wealth, drawdown, Sharpe) with 68%, 95%, and 99% confidence intervals
> * **Baseline Scorecard**: Assemble a comprehensive scorecard that benchmarks the classical approach for comparison in Sessions 2–4

## Setup, Data and Prerequisites
Load our packages and the baseline portfolio from the previous example.

In [ ]:
include("Include.jl");

Load the baseline portfolio and synthetic return data generated in the previous example.

In [ ]:
let
    # Load baseline portfolio and synthetic data -
    portfolio_data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    returns_data = load(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"));

    # Unpack into global-scope constants for use in later cells -
    global baseline_weights = portfolio_data["weights"];
    global μ_hat = portfolio_data["mu_hat"];
    global Σ_hat = portfolio_data["Sigma_hat"];
    global asset_names = portfolio_data["asset_names"];
    global df_returns = returns_data["returns"];
    global C_true = returns_data["C_true"];
    global N = length(asset_names);
    global R_target = 0.05 / 252;
    global bounds = hcat(zeros(N), ones(N));

    println("Loaded baseline portfolio with $(N) assets: $(asset_names)")
    println("Baseline weights: $(round.(baseline_weights .* 100, digits=1))%")
end

___
## Task 1: Stress Test — Correlation Shifts
We perturb the correlation structure toward a "crisis regime" where all assets become more correlated. Using the blending formula $\tilde{\mathbf{C}} = (1-\alpha)\mathbf{C} + \alpha\,\mathbf{1}\mathbf{1}^{\top}$, we test $\alpha \in \{0.0, 0.10, 0.25, 0.50\}$ and re-solve the optimization under each stressed covariance matrix.

> __What are we going to do?__ For each stress level $\alpha$, we blend the estimated correlation matrix toward the all-ones matrix (perfect correlation), reconstruct $\boldsymbol{\Sigma}$, solve the quadratic program, and record the new weights and portfolio volatility. Then we'll visualize the weight migration and eigenvalue collapse.
>
> __What should you see?__ As correlations increase, the diversification benefit shrinks. Portfolio variance _increases_ even though the optimizer is doing its best. The Bond becomes even more dominant — it's the only asset offering genuine diversification.

In [ ]:
let
    # Extract the estimated standard deviations and correlation from Σ_hat -
    σ_hat = sqrt.(diag(Σ_hat));
    D_inv = diagm(1.0 ./ σ_hat);
    C_hat = D_inv * Σ_hat * D_inv; # estimated correlation matrix
    D_mat = diagm(σ_hat);

    # Stress levels -
    αs = [0.0, 0.10, 0.25, 0.50];
    ones_matrix = ones(N, N);

    # Results storage -
    stress_results = DataFrame();

    for α in αs
        
        # Blend correlation toward crisis -
        C_stressed = (1 - α) .* C_hat .+ α .* ones_matrix;

        # Rebuild covariance matrix -
        Σ_stressed = D_mat * C_stressed * D_mat;

        # Solve -
        problem = build(MyPortfolioAllocationProblem;
            μ = μ_hat, Σ = Σ_stressed, bounds = bounds, R = R_target);
        result = solve_minvariance(problem);

        # Portfolio volatility under stress -
        vol_annual = round(sqrt(result.variance) * sqrt(252) * 100, digits=2);
        turnover = round(compute_turnover(baseline_weights, result.weights), digits=4);

        row = DataFrame(
            "α" => α,
            [name => round(w * 100, digits=1) for (name, w) in zip(asset_names, result.weights)]...,
            "Vol (%)" => vol_annual,
            "Turnover" => turnover
        );
        stress_results = vcat(stress_results, row);
    end

    println("Correlation Stress Test Results:")
    pretty_table(stress_results, tf=tf_markdown)
end

**Visualize:** Let's compare the baseline ($\alpha = 0$) and stressed ($\alpha = 0.5$) correlation matrices side by side, and examine how the eigenvalue spectrum collapses under stress.

In [ ]:
let
    # compute estimated correlation -
    σ_hat = sqrt.(diag(Σ_hat));
    D_inv = diagm(1.0 ./ σ_hat);
    C_hat = D_inv * Σ_hat * D_inv;
    D_mat = diagm(σ_hat);
    ones_matrix = ones(N, N);
    
    # baseline vs stressed correlation -
    α_stress = 0.5;
    C_stressed = (1 - α_stress) .* C_hat .+ α_stress .* ones_matrix;
    
    p1 = heatmap(C_hat, title="Baseline (α=0)", xticks=(1:N, asset_names),
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    p2 = heatmap(C_stressed, title="Stressed (α=0.5)", xticks=(1:N, asset_names),
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    
    # eigenvalue comparison across stress levels -
    αs_fine = [0.0, 0.1, 0.25, 0.5, 0.75];
    p3 = plot(xlabel="Principal Component", ylabel="Eigenvalue (×10⁴)",
        title="Eigenvalue Collapse Under Stress", legend=:topright);
    for α ∈ αs_fine
        C_s = (1 - α) .* C_hat .+ α .* ones_matrix;
        Σ_s = D_mat * C_s * D_mat;
        λ_s = eigvals(Σ_s) |> sort |> reverse;
        plot!(p3, 1:N, λ_s .* 10000, marker=:circle, ms=4, label="α=$(α)", lw=2);
    end
    
    plot(p1, p2, p3, layout=@layout([a b; c{0.6h}]), size=(900, 700))
end

___
## Task 2: Stress Test — Price Drops and Monte Carlo Simulation
We first compute the deterministic impact of sudden price drops, then use Monte Carlo simulation to generate 1,000 return paths and visualize the _distribution_ of portfolio outcomes with confidence bands.

> __What are we going to do?__ Two parts: (1) compute the portfolio-level loss from individual asset drops (deterministic), and (2) simulate 1,000 return paths from $\mathcal{N}(\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\Sigma}})$, compute cumulative wealth for each path, and plot the fan chart with 68%, 95%, and 99% confidence bands — the same approach used in the CHEME-5660 MAGBM examples.
>
> __What should you see?__ The heatmap reveals concentration risk. The Monte Carlo fan chart shows how _uncertain_ the portfolio's future is — the confidence bands widen over time, and the drawdown distribution gives us a probabilistic view of worst-case losses.

In [ ]:
let
    # Price drop scenarios -
    drop_levels = [-0.10, -0.20, -0.40];
    
    # Results -
    drop_results = DataFrame();

    for i in 1:N
        for drop in drop_levels
            
            # Portfolio loss = weight_i × drop
            portfolio_loss = baseline_weights[i] * drop * 100;

            row = DataFrame(
                "Asset" => asset_names[i],
                "Weight (%)" => round(baseline_weights[i] * 100, digits=1),
                "Drop (%)" => Int(drop * 100),
                "Portfolio Loss (%)" => round(portfolio_loss, digits=2)
            );
            drop_results = vcat(drop_results, row);
        end
    end

    println("Price Drop Stress Test (portfolio-level impact):")
    pretty_table(drop_results, tf=tf_markdown)
end

**Visualize:** Heatmap of portfolio losses across assets and drop levels.

> **What should you see?** A heatmap where the color intensity tracks the portfolio-level loss. The row corresponding to the highest-weight asset will show the deepest red — that's where the concentration risk lives.

In [ ]:
let
    # Build loss matrix: rows = assets, cols = drop levels -
    drop_levels = [0.10, 0.20, 0.40];
    loss_matrix = zeros(N, length(drop_levels));
    
    for (j, drop) in enumerate(drop_levels)
        for i in 1:N
            loss_matrix[i, j] = baseline_weights[i] * drop * 100;
        end
    end

    heatmap(["−10%", "−20%", "−40%"], asset_names, loss_matrix,
        xlabel="Price Drop", ylabel="Asset", title="Portfolio Loss (%) by Asset Drop",
        color=:Reds, size=(600, 400), clims=(0, maximum(loss_matrix)))
end

**Monte Carlo Simulation:** Now we simulate 1,000 return paths of length $T = 252$ days (1 year forward) from the estimated distribution $\mathcal{N}(\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\Sigma}})$. For each path, we compute the cumulative wealth of the min-variance portfolio and an equal-weight benchmark.

> __What should you see?__ A fan chart with gray individual paths, colored confidence bands (68%, 95%, 99%), and the expected wealth trajectory. The width of the bands at each time step quantifies the portfolio's uncertainty. Compare the min-variance bands to the equal-weight bands.

In [ ]:
let
    # Monte Carlo parameters -
    n_trials = 1000;
    T_sim = 252;  # 1 year forward
    
    # simulate return paths -
    dist = MvNormal(μ_hat, Σ_hat);
    
    # compute wealth for min-variance and equal-weight -
    w_equal = fill(1.0/N, N);
    mv_wealth = zeros(T_sim + 1, n_trials);
    ew_wealth = zeros(T_sim + 1, n_trials);
    mv_wealth[1, :] .= 1.0;
    ew_wealth[1, :] .= 1.0;
    
    for trial ∈ 1:n_trials
        for t ∈ 1:T_sim
            r_t = rand(dist);  # N-vector of returns
            mv_wealth[t+1, trial] = mv_wealth[t, trial] * (1.0 + dot(baseline_weights, r_t));
            ew_wealth[t+1, trial] = ew_wealth[t, trial] * (1.0 + dot(w_equal, r_t));
        end
    end
    
    # save for scorecard -
    global mc_mv_wealth = mv_wealth;
    global mc_ew_wealth = ew_wealth;
    
    # compute statistics at each time step -
    mv_mean = vec(mean(mv_wealth, dims=2));
    mv_std = vec(std(mv_wealth, dims=2));
    
    # plot fan chart -
    days = 0:T_sim;
    p = plot(title="Monte Carlo: Min-Variance Portfolio ($(n_trials) paths)",
        xlabel="Trading Day", ylabel="Cumulative Wealth (\$1 initial)",
        legend=:topleft, size=(750, 450));
    
    # individual paths (subset for visibility) -
    for i ∈ 1:min(200, n_trials)
        plot!(p, days, mv_wealth[:, i], color=:gray86, lw=0.5, alpha=0.3, label="");
    end
    
    # confidence bands -
    plot!(p, days, mv_mean .+ 2.576 .* mv_std, fillrange=mv_mean .- 2.576 .* mv_std,
        fillalpha=0.15, color=:steelblue, lw=0, label="99% CI");
    plot!(p, days, mv_mean .+ 1.96 .* mv_std, fillrange=mv_mean .- 1.96 .* mv_std,
        fillalpha=0.2, color=:steelblue, lw=0, label="95% CI");
    plot!(p, days, mv_mean .+ mv_std, fillrange=mv_mean .- mv_std,
        fillalpha=0.3, color=:steelblue, lw=0, label="68% CI");
    
    # expected value -
    plot!(p, days, mv_mean, lw=2.5, color=:darkblue, label="E[Wealth]");
    
    p
end

**Distribution of Outcomes:** Let's look at the terminal wealth and maximum drawdown distributions across the 1,000 simulated paths, comparing the min-variance portfolio to the equal-weight benchmark.

> __What should you see?__ The min-variance portfolio should have a _tighter_ terminal wealth distribution (lower variance) but a similar or lower median. The drawdown distributions reveal the tail risk — the worst-case losses across the simulated paths.

In [ ]:
let
    n_trials = size(mc_mv_wealth, 2);
    T_sim = size(mc_mv_wealth, 1) - 1;
    
    # terminal wealth -
    mv_terminal = mc_mv_wealth[end, :];
    ew_terminal = mc_ew_wealth[end, :];
    
    # max drawdown per path -
    mv_drawdowns = zeros(n_trials);
    ew_drawdowns = zeros(n_trials);
    for i ∈ 1:n_trials
        mv_peak = accumulate(max, mc_mv_wealth[:, i]);
        mv_dd = (mv_peak .- mc_mv_wealth[:, i]) ./ mv_peak;
        mv_drawdowns[i] = maximum(mv_dd);
        
        ew_peak = accumulate(max, mc_ew_wealth[:, i]);
        ew_dd = (ew_peak .- mc_ew_wealth[:, i]) ./ ew_peak;
        ew_drawdowns[i] = maximum(ew_dd);
    end
    
    # plot -
    p1 = histogram(mv_terminal, bins=50, alpha=0.6, label="Min-Variance", color=:steelblue,
        xlabel="Terminal Wealth (\$1 initial)", ylabel="Frequency", title="Terminal Wealth Distribution");
    histogram!(p1, ew_terminal, bins=50, alpha=0.5, label="Equal-Weight", color=:coral);
    vline!(p1, [median(mv_terminal)], lw=2, ls=:dash, color=:darkblue, label="MV median");
    vline!(p1, [median(ew_terminal)], lw=2, ls=:dash, color=:darkred, label="EW median");
    
    p2 = histogram(mv_drawdowns .* 100, bins=50, alpha=0.6, label="Min-Variance", color=:steelblue,
        xlabel="Max Drawdown (%)", ylabel="Frequency", title="Max Drawdown Distribution");
    histogram!(p2, ew_drawdowns .* 100, bins=50, alpha=0.5, label="Equal-Weight", color=:coral);
    
    plot(p1, p2, layout=(1,2), size=(1000, 400), legend=:topright)
end

___
## Task 3: Trading Cost Stress Test and Baseline Scorecard
We evaluate how portfolio performance changes as trading costs increase, then assemble a comprehensive scorecard that summarizes the baseline min-variance portfolio's performance — including Monte Carlo confidence intervals.

> __What are we going to do?__ First, we compute trading costs at 1×, 2×, 5×, and 10× the baseline assumption. Then we build the full scorecard with both point estimates (from the synthetic data) and Monte Carlo percentiles (from the 1,000 simulated paths). This scorecard is the quantitative benchmark that the AI rebalancing engine in Session 2 needs to beat.
>
> __What should you see?__ High-turnover portfolios suffer most from cost increases. The Monte Carlo percentiles add uncertainty quantification to every metric — giving us a range, not just a point estimate.

In [ ]:
let
    # Baseline cost assumption: 10 bps per unit traded -
    base_cost_bps = 10.0;
    cost_multipliers = [1, 2, 5, 10];
    
    # Starting point: equal-weight portfolio -
    w_equal = fill(1.0 / N, N);
    turnover = compute_turnover(w_equal, baseline_weights);

    # Results -
    cost_results = DataFrame();

    for mult in cost_multipliers
        cost_bps = base_cost_bps * mult;
        trading_cost = turnover * (cost_bps / 10000) * 100; # as % of portfolio

        row = DataFrame(
            "Cost Multiplier" => "$(mult)×",
            "Cost (bps/trade)" => Int(cost_bps),
            "Turnover" => round(turnover, digits=4),
            "Trading Cost (%)" => round(trading_cost, digits=4)
        );
        cost_results = vcat(cost_results, row);
    end

    println("Trading Cost Stress Test (equal-weight → min-variance rebalance):")
    pretty_table(cost_results, tf=tf_markdown)
end

**Baseline Scorecard:** Now we assemble the comprehensive scorecard, combining point estimates from the historical synthetic data with Monte Carlo percentiles from the simulated paths.

> __What should you see?__ A comparison table: Min-Variance vs. Equal-Weight across all metrics. The Monte Carlo columns show the 5th and 95th percentile outcomes — the "bad case" and "good case" scenarios.

In [ ]:
let
    # --- Point estimates from synthetic data ---
    R = Matrix(df_returns);
    portfolio_returns = R * baseline_weights;
    w_equal = fill(1.0 / N, N);
    eq_returns = R * w_equal;
    
    # metrics: min-variance -
    mv_ret = mean(portfolio_returns) * 252 * 100;
    mv_vol = std(portfolio_returns) * sqrt(252) * 100;
    mv_sharpe = mv_ret / mv_vol;
    mv_dd = compute_drawdown(portfolio_returns) * 100;
    mv_turnover = compute_turnover(w_equal, baseline_weights);
    mv_cost = mv_turnover * (10.0 / 10000) * 100;
    
    # metrics: equal-weight -
    ew_ret = mean(eq_returns) * 252 * 100;
    ew_vol = std(eq_returns) * sqrt(252) * 100;
    ew_sharpe = ew_ret / ew_vol;
    ew_dd = compute_drawdown(eq_returns) * 100;
    
    # --- Monte Carlo percentiles ---
    n_trials = size(mc_mv_wealth, 2);
    mv_terminal = mc_mv_wealth[end, :];
    ew_terminal = mc_ew_wealth[end, :];
    
    # MC drawdowns -
    mv_mc_dd = zeros(n_trials);
    ew_mc_dd = zeros(n_trials);
    for i ∈ 1:n_trials
        pk = accumulate(max, mc_mv_wealth[:, i]);
        mv_mc_dd[i] = maximum((pk .- mc_mv_wealth[:, i]) ./ pk) * 100;
        pk_e = accumulate(max, mc_ew_wealth[:, i]);
        ew_mc_dd[i] = maximum((pk_e .- mc_ew_wealth[:, i]) ./ pk_e) * 100;
    end
    
    # MC annualized return -
    mv_mc_ret = (mv_terminal .^ (252/252) .- 1.0) .* 100;
    ew_mc_ret = (ew_terminal .^ (252/252) .- 1.0) .* 100;
    
    # build comparison scorecard -
    scorecard = DataFrame(
        "Metric" => [
            "Expected Return (annual %)",
            "Volatility (annual %)",
            "Sharpe Ratio",
            "Max Drawdown (%)",
            "Turnover (vs equal-wt)",
            "Trading Cost (%, 10 bps)",
            "MC Return — 5th pctile (%)",
            "MC Return — 95th pctile (%)",
            "MC Max DD — 95th pctile (%)"
        ],
        "Min-Variance" => [
            round(mv_ret, digits=2),
            round(mv_vol, digits=2),
            round(mv_sharpe, digits=2),
            round(mv_dd, digits=2),
            round(mv_turnover, digits=4),
            round(mv_cost, digits=4),
            round(quantile(mv_mc_ret, 0.05), digits=2),
            round(quantile(mv_mc_ret, 0.95), digits=2),
            round(quantile(mv_mc_dd, 0.95), digits=2)
        ],
        "Equal-Weight" => [
            round(ew_ret, digits=2),
            round(ew_vol, digits=2),
            round(ew_sharpe, digits=2),
            round(ew_dd, digits=2),
            0.0,
            0.0,
            round(quantile(ew_mc_ret, 0.05), digits=2),
            round(quantile(ew_mc_ret, 0.95), digits=2),
            round(quantile(ew_mc_dd, 0.95), digits=2)
        ]
    );
    
    println("═"^65)
    println("  BASELINE SCORECARD: Min-Variance vs. Equal-Weight")
    println("═"^65)
    pretty_table(scorecard, tf = tf_markdown)
    
    # save for later sessions -
    save(joinpath(_PATH_TO_DATA, "baseline-scorecard.jld2"),
        Dict("scorecard" => scorecard, "portfolio_returns" => portfolio_returns,
             "baseline_weights" => baseline_weights,
             "mc_mv_wealth" => mc_mv_wealth, "mc_ew_wealth" => mc_ew_wealth));
end

**Visualize:** Cumulative wealth comparison using the _historical_ synthetic data, plus a grouped bar chart comparing Min-Variance vs. Equal-Weight across the key scorecard metrics.

> __What should you see?__ Two wealth curves starting at \$1.00. The min-variance portfolio should show lower volatility (smoother path) but potentially lower total return. The bar chart makes the trade-offs immediately visible.

In [ ]:
let
    # Compute cumulative wealth for both portfolios -
    R = Matrix(df_returns);
    w_equal = fill(1.0 / N, N);
    
    mv_returns = R * baseline_weights;
    mv_wealth = cumprod(1.0 .+ mv_returns);
    
    ew_returns = R * w_equal;
    ew_wealth = cumprod(1.0 .+ ew_returns);

    T_hist = length(mv_wealth);
    days = 1:T_hist;

    # cumulative wealth plot -
    p1 = plot(days, mv_wealth, label="Min-Variance", lw=2, color=:steelblue,
        xlabel="Trading Day", ylabel="Cumulative Wealth (\$1 initial)",
        title="Historical Synthetic Data", legend=:topleft);
    plot!(p1, days, ew_wealth, label="Equal-Weight", lw=2, color=:coral, linestyle=:dash);
    
    # scorecard comparison bar chart -
    mv_ret = mean(mv_returns) * 252 * 100;
    ew_ret = mean(ew_returns) * 252 * 100;
    mv_vol = std(mv_returns) * sqrt(252) * 100;
    ew_vol = std(ew_returns) * sqrt(252) * 100;
    mv_dd = compute_drawdown(mv_returns) * 100;
    ew_dd = compute_drawdown(ew_returns) * 100;
    
    metrics = ["Return (%)", "Volatility (%)", "Max DD (%)"];
    mv_vals = [mv_ret, mv_vol, mv_dd];
    ew_vals = [ew_ret, ew_vol, ew_dd];
    
    p2 = groupedbar(metrics, hcat(mv_vals, ew_vals),
        bar_position=:dodge, label=["Min-Variance" "Equal-Weight"],
        color=[:steelblue :coral], ylabel="Value",
        title="Scorecard Comparison", legend=:topright);
    
    plot(p1, p2, layout=(1,2), size=(1100, 400))
end

___
## Summary

> __Key Takeaways:__
>
> * **Correlation stress** increases portfolio volatility and further concentrates weights in the Bond — the eigenvalue spectrum collapses as $\alpha$ increases, meaning the optimizer has fewer dimensions to work with
> * **Price drop exposure** is dominated by the highest-weight asset — concentration risk in action. The heatmap makes this immediately visible
> * **Monte Carlo simulation** transforms our scorecard from point estimates into distributions — the 1,000 simulated paths provide 5th/95th percentile bounds on return, drawdown, and terminal wealth
> * **The baseline scorecard** (now with uncertainty quantification) gives us a comprehensive benchmark that the AI rebalancing engine in Session 2 must improve upon

The scorecard, Monte Carlo wealth paths, and portfolio data are saved to the `data/` directory for use in subsequent sessions.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.